In [2]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [3]:
organism = "homo_sapiens"
cell_source = "yolk_sac"
cell_state = None

reference = "GRCh38"

table_name = "clean_degs.xlsx" # local name

## Get sheet names

In [4]:
metadata = pd.read_json("../metadata.json")
sheet_names = [obj["sheet_name"] for obj in metadata["tables"]]

sheet_names

['T_17 DEGs scRNA-seq']

## Read in file

In [9]:
excel = pd.read_excel(table_name, sheet_name = sheet_names, skiprows = 2)

for e in list(excel.values()):
    print(e.columns)

Index(['cluster', 'gene', 'p_val_adj', 'logfc', 'Unnamed: 4', 'cluster.1',
       'gene.1', 'p_val_adj.1', 'logfc.1', 'Unnamed: 9',
       ...
       'Megakaryocytes_l', 'Progenitor_n', 'Progenitor_p', 'Progenitor_l',
       'migDC_n', 'migDC_p', 'migDC_l', 'pDC_n', 'pDC_p', 'pDC_l'],
      dtype='object', length=110)


In [10]:
cols = [
    {
        "sheet_name": sheet_name,
        "cols": {
            "logfc": "logfc",
            "gene": "feature_name",
            "cluster": "cell_type_label",
            "p_val_adj": "p_val",
            "p_val_adj.1": "p_corr"
        }
    } for sheet_name in sheet_names]

In [11]:
for idx, sheet in enumerate(cols):
    sname = sheet["sheet_name"]
    print(sname)
    excel_name = excel
    if idx < 2:
        excel_name = excel
    # only keep the columns we want
    excel_name[sname] = excel_name[sname].loc[:,sheet["cols"].keys()].rename(columns=sheet["cols"])
    # add sheet name to the dataframe
    excel_name[sname]["sheet_name"] = sname

T_17 DEGs scRNA-seq


In [12]:
df = pd.concat(list(excel.values()))

df.head()

,logfc,feature_name,cell_type_label,p_val,p_corr,sheet_name
0,9.608899,GJA5,AEC,0.0,4.617905e-202,T_17 DEGs scRNA-seq
1,9.158854,IL33,AEC,0.0,8.794971e-220,T_17 DEGs scRNA-seq
2,7.725796,CLEC14A,AEC,0.0,3.362997e-217,T_17 DEGs scRNA-seq
3,7.595031,TMEM100,AEC,0.0,5.134091e-212,T_17 DEGs scRNA-seq
4,7.540078,GJA4,AEC,0.0,5.963953e-219,T_17 DEGs scRNA-seq


## Unfiltered

In [13]:
unfiltered_df = df.copy()
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None 

result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({
        "extracted": {
            "organism": row["organism"], 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene"
            },
        "derived": {
            "organism": row["organism"], 
            "cell_type_id": None, 
            "cell_type_label": row["cell_type_label"], 
            "cell_source": row["cell_source"], 
            "cell_state": row["cell_state"], 
            "feature_name": row["feature_name"], 
            "feature_type": "gene", 
            "feature_identifier": row["feature_identifier"], 
            "feature_type": "ensembl"
            },
        "source": {
            "source_type": "deg", 
            "source_rationale": "unfiltered", 
            "source_id": row["sheet_name"],
            "source_metrics": {
                "p_corr": row["p_corr"],
                "log_fc": row["logfc"],
            }
            }
        })

result[:1]

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'AEC',
   'cell_source': 'yolk_sac',
   'cell_state': None,
   'feature_name': 'GJA5',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'AEC',
   'cell_source': 'yolk_sac',
   'cell_state': None,
   'feature_name': 'GJA5',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': 'T_17 DEGs scRNA-seq',
   'source_metrics': {'p_corr': 4.61790461862051e-202, 'log_fc': 9.608899}}}]

## Save unfiltered

In [14]:
with open("evidence_unfiltered_metrics.json", 'w') as f:
    json.dump(result, f, indent = 4)

## Perform the filtering

In [9]:
col_pval = "p_val_adj"
col_lfc = "logfc"
col_gene = "gene"
col_ct = "celltype"
col_rank = None

In [10]:
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

if col_rank:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)].sort_values(col_rank, ascending = True)
else:
    m = dfc[dfc[col_gene].isin(non_repeat_genes)]

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [11]:
markers[col_ct].value_counts()

celltype
Mesothelium               20
Endoderm                  20
MK                        20
Erythroid                 20
Early_Erythroid           18
Smooth_Muscle             15
Fibroblast                15
Macrophage                14
Microglia                 13
NK                        10
pDC precursor              9
HSPC_1                     9
Mac DC2                    8
HE                         7
Monocyte_Macrophage        7
Cycling DC2                7
ILC_precursor              6
VWF_EC                     6
Mono Mac DC2               6
Prolif_AEC                 5
Monocyte_ys_1              5
Mast_cell                  5
Eo_Basophil                4
Pre DC2                    3
Pre_Macrophage             3
HSPC_2                     3
AEC                        3
CMP                        2
Mono Mac pre DC2           2
Monocyte_0                 2
Lymphoid_progenitor        2
Sinusoidal_EC              2
MEMP                       1
LMPP                       1
Neutr

In [12]:
if col_rank:
    markers.groupby(col_ct)[col_rank].mean().sort_values()

## Find Ensembl ID

In [13]:
import sys
import time

from urllib.parse import urlparse, urlencode
from urllib.request import urlopen, Request
from urllib.error import HTTPError

In [14]:
class EnsemblRestClient(object):
    def __init__(self, server='http://rest.ensembl.org', reqs_per_sec=3):
        self.server = server
        self.reqs_per_sec = reqs_per_sec
        self.req_count = 0
        self.last_req = 0

    def perform_rest_action(self, endpoint, hdrs=None, params=None):
        if hdrs is None:
            hdrs = {}

        if 'Content-Type' not in hdrs:
            hdrs['Content-Type'] = 'application/json'

        if params:
            endpoint += '?' + urlencode(params)

        data = None

        # check if we need to rate limit ourselves
        if self.req_count >= self.reqs_per_sec:
            delta = time.time() - self.last_req
            if delta < 1:
                time.sleep(1 - delta)
            self.last_req = time.time()
            self.req_count = 0
        
        try:
            request = Request(self.server + endpoint, headers=hdrs)
            response = urlopen(request)
            content = response.read()
            if content:
                data = json.loads(content)
            self.req_count += 1

        except HTTPError as e:
            # check if we are being rate limited by the server
            if e.code == 429:
                if 'Retry-After' in e.headers:
                    retry = e.headers['Retry-After']
                    time.sleep(float(retry))
                    self.perform_rest_action(endpoint, hdrs, params)
            else:
                sys.stderr.write('Request failed for {0}: Status code: {1.code} Reason: {1.reason}\n'.format(endpoint, e))
           
        return data

    def get_id(self, species, symbol):
        genes = self.perform_rest_action(
            endpoint='/xrefs/symbol/{0}/{1}'.format(species, symbol), 
            params={'object_type': 'gene'}
        )
        if genes:
            stable_id = genes[0]['id']
            return stable_id
        else:
            return "gene not found"

In [15]:
def run(symbol):
    client = EnsemblRestClient()
    gene_id = client.get_id('human', symbol)
    return gene_id

## Convert to evidence json


In [16]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = df["feature_name"]
df["feature_identifier"] = df["feature_identifier"].apply(run)
result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

In [17]:
result

[{'cell_type_label': 'AEC',
  'gene': 'ADGRL4',
  'organism': 'homo_sapiens',
  'cell_source': 'yolk_sac',
  'cell_state': None,
  'gene_id': 'ENSG00000162618'},
 {'cell_type_label': 'AEC',
  'gene': 'MYCT1',
  'organism': 'homo_sapiens',
  'cell_source': 'yolk_sac',
  'cell_state': None,
  'gene_id': 'ENSG00000120279'},
 {'cell_type_label': 'AEC',
  'gene': 'C10orf10',
  'organism': 'homo_sapiens',
  'cell_source': 'yolk_sac',
  'cell_state': None,
  'gene_id': 'ENSG00000165507'},
 {'cell_type_label': 'CMP',
  'gene': 'NUCB2',
  'organism': 'homo_sapiens',
  'cell_source': 'yolk_sac',
  'cell_state': None,
  'gene_id': 'ENSG00000070081'},
 {'cell_type_label': 'CMP',
  'gene': 'VAMP8',
  'organism': 'homo_sapiens',
  'cell_source': 'yolk_sac',
  'cell_state': None,
  'gene_id': 'ENSG00000118640'},
 {'cell_type_label': 'Cycling DC2',
  'gene': 'CLEC10A',
  'organism': 'homo_sapiens',
  'cell_source': 'yolk_sac',
  'cell_state': None,
  'gene_id': 'ENSG00000132514'},
 {'cell_type_label':

## Save evidence

In [18]:
with open("evidence.json", "w") as f:
    json.dump(result, f, indent = 4) 